# Browserless: browser automation via MCP

<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/tools/browserless.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Browserless](https://browserless.io) runs real browsers in the cloud and exposes them through a hosted MCP server at `https://mcp.browserless.io/mcp`. With `llama-index-tools-mcp` you can load its tools as LlamaIndex tools without a partner package:

- `browserless_smartscraper`: scrape one page as markdown, HTML, links, a screenshot, a PDF, or AI-extracted JSON
- `browserless_search`: search the web and optionally scrape each result
- `browserless_map` and `browserless_crawl`: list a site's URLs, or crawl the site and scrape every page
- `browserless_export`, `browserless_performance`, `browserless_function`: export a page in its native format, run a Lighthouse audit, or run your own Puppeteer code
- `browserless_agent`: drive one browser step by step (`goto`, `snapshot`, `click`, `type`, `evaluate`, ...)
- `browserless_skill`: load tuned, site-specific recipes for `browserless_agent`

The server also exposes account tools and widget-only tools for MCP Apps clients. This notebook loads only the browsing tools.

## Setup

In [ ]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

### Credentials

Get a Browserless API token at [account.browserless.io](https://account.browserless.io). The agent examples also use Anthropic, so set `ANTHROPIC_API_KEY` to run those cells.

In [ ]:
import getpass
import os

if not os.environ.get("BROWSERLESS_TOKEN"):
    os.environ["BROWSERLESS_TOKEN"] = getpass.getpass("BROWSERLESS_TOKEN:\n")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY:\n")

## Connect and load tools

Two details:

- The default 30-second `timeout` is too short for the first call's MCP handshake plus browser warm-up, so pass `timeout=120`.
- Browserless tool schemas include an optional `_prompt` field. Pydantic rejects field names that start with an underscore, so `McpToolSpec` cannot build a schema for those tools. The small subclass below skips such fields when it builds each tool's schema.

In [ ]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec


class BrowserlessToolSpec(McpToolSpec):
    def create_model_from_json_schema(self, schema, model_name="DynamicModel"):
        properties = {
            name: prop
            for name, prop in schema.get("properties", {}).items()
            if not name.startswith("_")
        }
        return super().create_model_from_json_schema(
            {**schema, "properties": properties}, model_name
        )


client = BasicMCPClient(
    "https://mcp.browserless.io/mcp",
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)

tools = await BrowserlessToolSpec(
    client=client,
    allowed_tools=[
        "browserless_smartscraper",
        "browserless_search",
        "browserless_map",
        "browserless_crawl",
        "browserless_export",
        "browserless_performance",
        "browserless_function",
        "browserless_agent",
        "browserless_skill",
    ],
).to_tool_list_async()

for t in tools:
    print(t.metadata.name)

## Call a tool directly

Each tool is a regular LlamaIndex `FunctionTool`, so you can call it without an LLM in the loop. `raw_output` holds the MCP result; `text()` joins its text parts.

In [ ]:
def text(output):
    return "\n".join(
        c.text for c in output.raw_output.content if c.type == "text"
    )


smartscraper = next(
    t for t in tools if t.metadata.name == "browserless_smartscraper"
)
result = await smartscraper.acall(
    url="https://example.com", formats=["markdown"]
)
print(text(result)[:400])

## Use the tools in an agent

Hand the tools to a `FunctionAgent` powered by Claude. The model picks which tool to call at each step.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

agent = FunctionAgent(
    name="BrowserlessAgent",
    description="Browses and scrapes the web via the Browserless MCP server.",
    llm=Anthropic(model="claude-sonnet-5-5"),
    tools=tools,
    system_prompt="You are a research assistant. Use the Browserless tools to scrape and summarize the web.",
)

response = await agent.run(
    "Scrape https://news.ycombinator.com and list the top 5 headlines as a markdown bullet list."
)
print(response)

## Keep one browser across calls

`browserless_agent` drives a single browser across several calls. Each call is one-shot by default: the browser closes when the call finishes. To keep it, pass `keepSessionAlive=True` on the first call, then pass the `sessionId` from its reply on every later call. Send `close` when you are done, because an open browser holds one of your account's concurrent sessions until it idles out.

The `sessionId` identifies the browser on its own, so it works even though `BasicMCPClient` opens a new MCP connection for every tool call.

In [ ]:
import re

browser = next(t for t in tools if t.metadata.name == "browserless_agent")

first = await browser.acall(
    method="goto", params={"url": "https://example.com"}, keepSessionAlive=True
)
session_id = re.search(r"sessionId: (\S+)", text(first)).group(1)

title = await browser.acall(
    method="evaluate",
    params={"content": "document.title"},
    sessionId=session_id,
)
print(text(title).split("\n\nsessionId")[0])

await browser.acall(method="close", sessionId=session_id)

## Let the LLM drive the browser

The `browserless_agent` tool description tells the model to set `keepSessionAlive`, pass the `sessionId` back, and close the browser at the end, so the agent from above can handle a multi-step task as is.

In [ ]:
response = await agent.run(
    "Use browserless_agent to open https://news.ycombinator.com, click the first story link, "
    "and report the title and first paragraph of that page."
)
print(response)

## Resources

- Browserless MCP server setup: [docs.browserless.io/mcp/browserless-mcp-server/setup](https://docs.browserless.io/mcp/browserless-mcp-server/setup)
- Browser agent reference: [docs.browserless.io/mcp/browser-agent](https://docs.browserless.io/mcp/browser-agent)
- Runnable cookbook with these examples: [`browserless/browserless-llamaindex`](https://github.com/browserless/browserless-llamaindex)
- LlamaIndex MCP adapter: [`llama-index-tools-mcp`](https://github.com/run-llama/llama_index/tree/main/llama-index-integrations/tools/llama-index-tools-mcp)